In [595]:
!pip install pyserial

In [596]:
import serial, time
!pip install pyserial

In [597]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [598]:
print(serial)

<module 'serial' from 'C:\\Users\\boome\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [599]:
print(serial.__file__)

C:\Users\boome\anaconda3\Lib\site-packages\serial\__init__.py


In [600]:
print(serial.__version__)

3.5


In [601]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [602]:
baudrate = 115200

In [603]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [604]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [605]:
ser.in_waiting

0

In [606]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [607]:
read_all(ser)

'dual servo control over serial\n'

In [608]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [609]:
read_one_line(ser)

''

In [610]:
read_all(ser)

''

In [611]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [612]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

In [613]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

## Parameters
![Screenshot 2026-06-19 234816.png](attachment:21bb921d-4bfa-4f88-9196-fb7a3789d1a5.png)

In [614]:
# measure based off sketch parameters
# units of cm
A = 11.5     # height of base of l1
B = 3    # offset from z1 to l1|
C = 5.5      # offset from l1 to l2
D = 1.5     # offset from l2 to center of EOAT
l1 = 24   # base link
l2 = 21.5 # tip link
l3 = 21   # gripper link

gripper_open = 1500
gripper_closed = 1050

################ for reference
#T01 = DH(0,0,th1+90,A)
#T12 = DH(90,0,th2+90,B)
#T23 = DH(180,l1,th3+90,C)
#T34 = DH(180,l2,th4-90,-D)

## Get from x,y,z postions to arduino servo values

In [615]:
# takes 3d coordinant and outputs required servo angles
def inverseKinematics(X, Y, Z):

    #define offset between base and grabber
    offset = B-C-D
    
    #define position as being relative to base
    P_wrist_0 = np.array([X,Y,Z+l3,1]) 

    #planer distance from offset to wrist
    planer_dist = np.sqrt(P_wrist_0[0]**2 + P_wrist_0[1]**2 - offset**2)

    #solve for th1.
    th1 = -np.arctan2(-planer_dist*P_wrist_0[0] + offset*P_wrist_0[1], offset*P_wrist_0[0] + planer_dist*P_wrist_0[1])*rtd + 90

    #distance from origin1 to tip
    r_squared = planer_dist**2 + (P_wrist_0[2]-A)**2
    
    #law of cos for angle between links
    alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
    sin_alpha_p = np.sqrt(1-alpha_temp**2)
    alpha = np.arctan2(sin_alpha_p, alpha_temp)
    
    #vertical angle theorem for theta 2
    theta3 = 180 - alpha*rtd
    
    #triangle in link1 co-ordinant system for psi
    psi = np.arctan2(l2*sind(theta3), l1+l2*cosd(theta3))
    
    #angle of r to x-axis
    beta = np.arctan2(P_wrist_0[2]-A, planer_dist)
    
    #difference in beta and psi is theta 1
    theta2 = (beta + psi)*rtd

    th2 = theta2
    th3 = theta3 

    # l3 should be perpendicular to the ground at all times 
    th4 = th3 - th2 + 90

    return th1, th2, th3, th4

In [616]:
def thetaInterpolation(th1, th2, th3, th4):
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 1000    # minimum servo value
    max_new = 2000    # maximum servo value

    #linear interpolate for the first theta value (min and max flipped)
    myint = max_new + ((th1-theta_min)*(min_new-max_new))/(theta_max-theta_min)

    #linear interpolate for the second theta value
    myint2 = min_new + ((th2-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    #third theta (need to adjust once tested)
    myint3 = min_new + ((th3-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    #fourth theta (need to adjust once tested)
    myint4 = min_new + ((th4-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    return myint, myint2, myint3, myint4

In [617]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

### Test

In [721]:
test_xyz = np.array([B-C-D,l2,A+l1-l3]) #home position
test_xyz

array([-4. , 21.5, 14.5])

In [638]:
test_angles = inverseKinematics(test_xyz[0],test_xyz[1],test_xyz[2])
#test_angles = inverseKinematics(20, 20, 0)
test_angles

(np.float64(90.0), np.float64(90.0), np.float64(90.0), np.float64(90.0))

In [639]:
T01 = DH(0,0,180-test_angles[0],A)
T12 = DH(90,0,test_angles[1],B) 
T23 = DH(180,l1,test_angles[2],C)
T34 = DH(180,l2,test_angles[3]-180,-D)

T04 = T01@T12@T23@T34
P_tip_4 = np.array([l3,0,0,1])

P_tip_0_check = T04 @ P_tip_4
prettymat(P_tip_0_check)

array([-4. , 21.5, 14.5,  1. ])

In [640]:
test_servo = thetaInterpolation(test_angles[0],test_angles[1],test_angles[2],test_angles[3])
test_servo

(np.float64(1500.0),
 np.float64(1500.0),
 np.float64(1500.0),
 np.float64(1500.0))

In [641]:
byte1_test, byte2_test = break_into_two(test_servo[0])
byte3_test, byte4_test = break_into_two(test_servo[1])
byte5_test, byte6_test = break_into_two(test_servo[2])
byte7_test, byte8_test = break_into_two(test_servo[3])
byte9_test, byte10_test = break_into_two(gripper_closed)

In [713]:
WriteByte(ser, int(byte1_test))   # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2_test))   # servo 1 LSB
time.sleep(0.05)
WriteByte(ser, int(byte3_test))   # servo 2 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4_test))   # servo 2 LSB
time.sleep(0.05)
WriteByte(ser, int(byte5_test))   # servo 3 LSB
time.sleep(0.05)
WriteByte(ser, int(byte6_test))   # servo 3 LSB
time.sleep(0.05)
WriteByte(ser, int(byte7_test))   # servo 4 LSB
time.sleep(0.05)
WriteByte(ser, int(byte8_test))   # servo 4 LSB
time.sleep(0.05)
WriteByte(ser, int(byte9_test))   # servo 5 LSB
time.sleep(0.05)
WriteByte(ser, int(byte10_test))   # servo 5 LSB
time.sleep(0.05)

### Loop

In [714]:
homeX = B-C-D
homeY = l2
homeZ = A+l1-l3
#define pick location
pickX = 16
pickY = 24
pickZ = 0

#define place location
placeX = -20
placeY = 20
placeZ = 0

#define obstical location
obstacleX = 0
obstacleY = 10*2.54

lift = 10  # z displacement from pick
obstacle_radius = 2 * 2.45  # 2 inches in cm
clearance = 5 # buffer distance from obstacle (cm)

#number of positions for each path
N1 = 3  # z  down into the pick
N2 = 2  # z  up above the place z
N3 = 2  # y  to below the obstacle
N4 = 10 # x  across to the place x
N5 = 5  # y  to the place y
N6 = 3  # z  down to the place
N7 = 2  # z  retract up 
N8 = 3  # return home

z_clear = max(pickZ, placeZ) + lift                          # travel height: above place z (and above pick)
y_safe  = obstacleY - obstacle_radius - clearance            # a lane in front of (below) the obstacle

above_pick = [pickX, pickY, z_clear]                        # start: positioned above the pick
path1 = np.linspace(above_pick, [pickX, pickY, pickZ], N1)   # z  down into the pick
# ---- PAUSE: close gripper ----
path2 = np.linspace([pickX,  pickY,  pickZ],   [pickX,  pickY,  z_clear], N2)  # z  up above the place z
path3 = np.linspace([pickX,  pickY,  z_clear], [pickX,  y_safe, z_clear], N3)  # y  to below the obstacle
path4 = np.linspace([pickX,  y_safe, z_clear], [placeX, y_safe, z_clear], N4)  # x  across to the place x
path5 = np.linspace([placeX, y_safe, z_clear], [placeX, placeY, z_clear], N5)  # y  to the place y
path6 = np.linspace([placeX, placeY, z_clear], [placeX, placeY, placeZ],  N6)  # z  down to the place
# ---- PAUSE: open gripper ----
path7 = np.linspace([placeX, placeY, placeZ],  [placeX, placeY, z_clear], N7)  # z  retract up 
path8 = np.linspace([placeX, placeY, z_clear], [homeX,  homeY,  homeZ],   N8)  # return to home

path1

array([[16., 24., 10.],
       [16., 24.,  5.],
       [16., 24.,  0.]])

In [715]:
# run each path through inverseKinemnatics function to get angles for each position
path1_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path1])
path2_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path2])
path3_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path3])
path4_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path4])
path5_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path5])
path6_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path6])
path7_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path7])
path8_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path8])

path8_angles

array([[53.13010235, 73.56559993, 82.98708766, 99.42148773],
       [69.56481211, 84.64170388, 90.367817  , 95.72611312],
       [90.        , 90.        , 90.        , 90.        ]])

In [716]:
# run each path_angles through thetaInterpolation function to get angles for each position in servo language
path1_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path1_angles])
path2_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path2_angles])
path3_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path3_angles])
path4_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path4_angles])
path5_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path5_angles])
path6_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path6_angles])
path7_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path7_angles])
path8_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path8_angles])

path8_servo

array([[1704.8327647 , 1408.69777742, 1461.03937591, 1552.3415985 ],
       [1613.52882161, 1470.23168821, 1502.04342779, 1531.81173958],
       [1500.        , 1500.        , 1500.        , 1500.        ]])

In [717]:
# add in gripper control
path1_full = np.column_stack((path1_servo, np.full(len(path1),gripper_open))) ## open gripper
path2_full = np.column_stack((path2_servo, np.full(len(path2),gripper_closed))) ## closed gripper
path3_full = np.column_stack((path3_servo, np.full(len(path3),gripper_closed))) ## closed gripper
path4_full = np.column_stack((path4_servo, np.full(len(path4),gripper_closed))) ## closed gripper
path5_full = np.column_stack((path5_servo, np.full(len(path5),gripper_closed))) ## closed gripper
path6_full = np.column_stack((path6_servo, np.full(len(path6),gripper_closed))) ## closed gripper
path7_full = np.column_stack((path7_servo, np.full(len(path7),gripper_open))) ## open gripper
path8_full = np.column_stack((path8_servo, np.full(len(path8),gripper_open))) ## open gripper

path8_full

array([[1704.8327647 , 1408.69777742, 1461.03937591, 1552.3415985 ,
        1500.        ],
       [1613.52882161, 1470.23168821, 1502.04342779, 1531.81173958,
        1500.        ],
       [1500.        , 1500.        , 1500.        , 1500.        ,
        1500.        ]])

## path 1

In [709]:
#define arrays to hold the four bytes
byte1_1 = np.zeros(len(path1_full), dtype=int)
byte2_1 = np.zeros(len(path1_full), dtype=int)
byte3_1 = np.zeros(len(path1_full), dtype=int)
byte4_1 = np.zeros(len(path1_full), dtype=int)
byte5_1 = np.zeros(len(path1_full), dtype=int)
byte6_1 = np.zeros(len(path1_full), dtype=int)
byte7_1 = np.zeros(len(path1_full), dtype=int)
byte8_1 = np.zeros(len(path1_full), dtype=int)
byte9_1 = np.zeros(len(path1_full), dtype=int)
byte10_1 = np.zeros(len(path1_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path1_full)):
    byte1_1[i], byte2_1[i] = break_into_two(path1_full[i,0])
    byte3_1[i], byte4_1[i] = break_into_two(path1_full[i,1])
    byte5_1[i], byte6_1[i] = break_into_two(path1_full[i,2])
    byte7_1[i], byte8_1[i] = break_into_two(path1_full[i,3])
    byte9_1[i], byte10_1[i] = break_into_two(path1_full[i,4])

print(byte1_1,'\n\n',byte2_1,'\n\n\n',
      byte3_1,'\n\n',byte4_1,'\n\n\n',
      byte5_1,'\n\n',byte6_1,'\n\n\n',
      byte7_1,'\n\n',byte8_1,'\n\n\n',
      byte9_1,'\n\n',byte10_1)

[4 4 4] 

 [244 244 244] 


 [5 5 5] 

 [121 103  72] 


 [5 5 6] 

 [171 223   4] 


 [6 6 6] 

 [ 13  84 152] 


 [5 5 5] 

 [220 220 220]


In [710]:
# Send all path points to both servos
x = 1
for i in range(len(path1_full)):
    WriteByte(ser, int(byte1_1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_1[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_1[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_1[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_1[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_1[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_1[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_1[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_1[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_1[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_1[i]*256 + byte2_1[i]}  "
          f"th2={byte3_1[i]*256 + byte4_1[i]}  "
          f"th3={byte5_1[i]*256 + byte6_1[i]}  "
          f"th4={byte7_1[i]*256 + byte8_1[i]}  "
          f"th5={byte9_1[i]*256 + byte10_1[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1268  th2=1401  th3=1451  th4=1549  th5=1500
Step  1:  th1=1268  th2=1383  th3=1503  th4=1620  th5=1500
Step  2:  th1=1268  th2=1352  th3=1540  th4=1688  th5=1500


## path 2

In [711]:
#define arrays to hold the four bytes
byte1_2 = np.zeros(len(path2_full), dtype=int)
byte2_2 = np.zeros(len(path2_full), dtype=int)
byte3_2 = np.zeros(len(path2_full), dtype=int)
byte4_2 = np.zeros(len(path2_full), dtype=int)
byte5_2 = np.zeros(len(path2_full), dtype=int)
byte6_2 = np.zeros(len(path2_full), dtype=int)
byte7_2 = np.zeros(len(path2_full), dtype=int)
byte8_2 = np.zeros(len(path2_full), dtype=int)
byte9_2 = np.zeros(len(path2_full), dtype=int)
byte10_2 = np.zeros(len(path2_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path2_full)):
    byte1_2[i], byte2_2[i] = break_into_two(path2_full[i,0])
    byte3_2[i], byte4_2[i] = break_into_two(path2_full[i,1])
    byte5_2[i], byte6_2[i] = break_into_two(path2_full[i,2])
    byte7_2[i], byte8_2[i] = break_into_two(path2_full[i,3])
    byte9_2[i], byte10_2[i] = break_into_two(path2_full[i,4])

print(byte1_2,'\n\n',byte2_2,'\n\n\n',
      byte3_2,'\n\n',byte4_2,'\n\n\n',
      byte5_2,'\n\n',byte6_2,'\n\n\n',
      byte7_2,'\n\n',byte8_2,'\n\n\n',
      byte9_2,'\n\n',byte10_2)


[4 4] 

 [244 244] 


 [5 5] 

 [ 72 121] 


 [6 5] 

 [  4 171] 


 [6 6] 

 [152  13] 


 [4 4] 

 [26 26]


In [712]:
# Send all path points to both servos
x = 1
for i in range(len(path2_full)):
    WriteByte(ser, int(byte1_2[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_2[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_2[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_2[i]))   # servo 3 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_2[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_2[i]))   # servo 4 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_2[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_2[i]))   # servo 5 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_2[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_2[i]*256 + byte2_2[i]}  "
          f"th2={byte3_2[i]*256 + byte4_2[i]}  "
          f"th3={byte5_2[i]*256 + byte6_2[i]}  "
          f"th4={byte7_2[i]*256 + byte8_2[i]}  "
          f"th5={byte9_2[i]*256 + byte10_2[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1268  th2=1352  th3=1540  th4=1688  th5=1050
Step  1:  th1=1268  th2=1401  th3=1451  th4=1549  th5=1050


## path 3

In [679]:
#define arrays to hold the four bytes
byte1_3 = np.zeros(len(path3_full), dtype=int)
byte2_3 = np.zeros(len(path3_full), dtype=int)
byte3_3 = np.zeros(len(path3_full), dtype=int)
byte4_3 = np.zeros(len(path3_full), dtype=int)
byte5_3 = np.zeros(len(path3_full), dtype=int)
byte6_3 = np.zeros(len(path3_full), dtype=int)
byte7_3 = np.zeros(len(path3_full), dtype=int)
byte8_3 = np.zeros(len(path3_full), dtype=int)
byte9_3 = np.zeros(len(path3_full), dtype=int)
byte10_3 = np.zeros(len(path3_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path3_full)):
    byte1_3[i], byte2_3[i] = break_into_two(path3_full[i,0])
    byte3_3[i], byte4_3[i] = break_into_two(path3_full[i,1])
    byte5_3[i], byte6_3[i] = break_into_two(path3_full[i,2])
    byte7_3[i], byte8_3[i] = break_into_two(path3_full[i,3])
    byte9_3[i], byte10_3[i] = break_into_two(path3_full[i,4])

print(byte1_3,'\n\n',byte2_3,'\n\n\n',
      byte3_3,'\n\n',byte4_3,'\n\n\n',
      byte5_3,'\n\n',byte6_3,'\n\n\n',
      byte7_3,'\n\n',byte8_3,'\n\n\n',
      byte9_3,'\n\n',byte10_3)

[4 4] 

 [244 163] 


 [5 5] 

 [121 208] 


 [5 6] 

 [171  19] 


 [6 6] 

 [13 30] 


 [4 4] 

 [26 26]


In [680]:
# Send all path points to both servos
x = 1
for i in range(len(path3_full)):
    WriteByte(ser, int(byte1_3[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_3[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_3[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_3[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_3[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_3[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_3[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_3[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_3[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_3[i]*256 + byte2_3[i]}  "
          f"th2={byte3_3[i]*256 + byte4_3[i]}  "
          f"th3={byte5_3[i]*256 + byte6_3[i]}  "
          f"th4={byte7_3[i]*256 + byte8_3[i]}  "
          f"th5={byte9_3[i]*256 + byte10_3[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1268  th2=1401  th3=1451  th4=1549  th5=1050
Step  1:  th1=1187  th2=1488  th3=1555  th4=1566  th5=1050


## path 4

In [681]:
#define arrays to hold the four bytes
byte1_4 = np.zeros(len(path4_full), dtype=int)
byte2_4 = np.zeros(len(path4_full), dtype=int)
byte3_4 = np.zeros(len(path4_full), dtype=int)
byte4_4 = np.zeros(len(path4_full), dtype=int)
byte5_4 = np.zeros(len(path4_full), dtype=int)
byte6_4 = np.zeros(len(path4_full), dtype=int)
byte7_4 = np.zeros(len(path4_full), dtype=int)
byte8_4 = np.zeros(len(path4_full), dtype=int)
byte9_4 = np.zeros(len(path4_full), dtype=int)
byte10_4 = np.zeros(len(path4_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path4_full)):
    byte1_4[i], byte2_4[i] = break_into_two(path4_full[i,0])
    byte3_4[i], byte4_4[i] = break_into_two(path4_full[i,1])
    byte5_4[i], byte6_4[i] = break_into_two(path4_full[i,2])
    byte7_4[i], byte8_4[i] = break_into_two(path4_full[i,3])
    byte9_4[i], byte10_4[i] = break_into_two(path4_full[i,4])

print(byte1_4,'\n\n',byte2_4,'\n\n\n',
      byte3_4,'\n\n',byte4_4,'\n\n\n',
      byte5_4,'\n\n',byte6_4,'\n\n\n',
      byte7_4,'\n\n',byte8_4,'\n\n\n',
      byte9_4,'\n\n',byte10_4)

[4 4 4 5 5 5 6 6 6 6] 

 [163 200 250  59 136 220  42 108 161 203] 


 [5 5 6 6 6 6 6 5 5 5] 

 [208 244  18  39  47  39  18 244 208 168] 


 [6 6 6 6 6 6 6 6 6 5] 

 [ 19  54  80  96 102  96  80  54  19 230] 


 [6 6 6 6 6 6 6 6 6 6] 

 [30 30 25 20 18 20 25 30 30 26] 


 [4 4 4 4 4 4 4 4 4 4] 

 [26 26 26 26 26 26 26 26 26 26]


In [682]:
# Send all path points to both servos
x = 1
for i in range(len(path4_full)):
    WriteByte(ser, int(byte1_4[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_4[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_4[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_4[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_4[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_4[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_4[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_4[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_4[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_4[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_4[i]*256 + byte2_4[i]}  "
          f"th2={byte3_4[i]*256 + byte4_4[i]}  "
          f"th3={byte5_4[i]*256 + byte6_4[i]}  "
          f"th4={byte7_4[i]*256 + byte8_4[i]}  "
          f"th5={byte9_4[i]*256 + byte10_4[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1187  th2=1488  th3=1555  th4=1566  th5=1050
Step  1:  th1=1224  th2=1524  th3=1590  th4=1566  th5=1050
Step  2:  th1=1274  th2=1554  th3=1616  th4=1561  th5=1050
Step  3:  th1=1339  th2=1575  th3=1632  th4=1556  th5=1050
Step  4:  th1=1416  th2=1583  th3=1638  th4=1554  th5=1050
Step  5:  th1=1500  th2=1575  th3=1632  th4=1556  th5=1050
Step  6:  th1=1578  th2=1554  th3=1616  th4=1561  th5=1050
Step  7:  th1=1644  th2=1524  th3=1590  th4=1566  th5=1050
Step  8:  th1=1697  th2=1488  th3=1555  th4=1566  th5=1050
Step  9:  th1=1739  th2=1448  th3=1510  th4=1562  th5=1050


## path 5

In [683]:
#define arrays to hold the four bytes
byte1_5 = np.zeros(len(path5_full), dtype=int)
byte2_5 = np.zeros(len(path5_full), dtype=int)
byte3_5 = np.zeros(len(path5_full), dtype=int)
byte4_5 = np.zeros(len(path5_full), dtype=int)
byte5_5 = np.zeros(len(path5_full), dtype=int)
byte6_5 = np.zeros(len(path5_full), dtype=int)
byte7_5 = np.zeros(len(path5_full), dtype=int)
byte8_5 = np.zeros(len(path5_full), dtype=int)
byte9_5 = np.zeros(len(path5_full), dtype=int)
byte10_5 = np.zeros(len(path5_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path5_full)):
    byte1_5[i], byte2_5[i] = break_into_two(path5_full[i,0])
    byte3_5[i], byte4_5[i] = break_into_two(path5_full[i,1])
    byte5_5[i], byte6_5[i] = break_into_two(path5_full[i,2])
    byte7_5[i], byte8_5[i] = break_into_two(path5_full[i,3])
    byte9_5[i], byte10_5[i] = break_into_two(path5_full[i,4])

print(byte1_5,'\n\n',byte2_5,'\n\n\n',
      byte3_5,'\n\n',byte4_5,'\n\n\n',
      byte5_5,'\n\n',byte6_5,'\n\n\n',
      byte7_5,'\n\n',byte8_5,'\n\n\n',
      byte9_5,'\n\n',byte10_5)

[6 6 6 6 6] 

 [203 194 185 176 168] 


 [5 5 5 5 5] 

 [168 158 149 139 128] 


 [5 5 5 5 5] 

 [230 219 207 194 181] 


 [6 6 6 6 6] 

 [26 24 22 19 16] 


 [4 4 4 4 4] 

 [26 26 26 26 26]


In [684]:
# Send all path points to both servos
x = 1
for i in range(len(path5_full)):
    WriteByte(ser, int(byte1_5[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_5[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_5[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_5[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_5[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_5[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_5[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_5[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_5[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_5[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_5[i]*256 + byte2_5[i]}  "
          f"th2={byte3_5[i]*256 + byte4_5[i]}  "
          f"th3={byte5_5[i]*256 + byte6_5[i]}  "
          f"th4={byte7_5[i]*256 + byte8_5[i]}  "
          f"th5={byte9_5[i]*256 + byte10_5[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1739  th2=1448  th3=1510  th4=1562  th5=1050
Step  1:  th1=1730  th2=1438  th3=1499  th4=1560  th5=1050
Step  2:  th1=1721  th2=1429  th3=1487  th4=1558  th5=1050
Step  3:  th1=1712  th2=1419  th3=1474  th4=1555  th5=1050
Step  4:  th1=1704  th2=1408  th3=1461  th4=1552  th5=1050


## path 6

In [685]:
#define arrays to hold the four bytes
byte1_6 = np.zeros(len(path6_full), dtype=int)
byte2_6 = np.zeros(len(path6_full), dtype=int)
byte3_6 = np.zeros(len(path6_full), dtype=int)
byte4_6 = np.zeros(len(path6_full), dtype=int)
byte5_6 = np.zeros(len(path6_full), dtype=int)
byte6_6 = np.zeros(len(path6_full), dtype=int)
byte7_6 = np.zeros(len(path6_full), dtype=int)
byte8_6 = np.zeros(len(path6_full), dtype=int)
byte9_6 = np.zeros(len(path6_full), dtype=int)
byte10_6 = np.zeros(len(path6_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path6_full)):
    byte1_6[i], byte2_6[i] = break_into_two(path6_full[i,0])
    byte3_6[i], byte4_6[i] = break_into_two(path6_full[i,1])
    byte5_6[i], byte6_6[i] = break_into_two(path6_full[i,2])
    byte7_6[i], byte8_6[i] = break_into_two(path6_full[i,3])
    byte9_6[i], byte10_6[i] = break_into_two(path6_full[i,4])

print(byte1_6,'\n\n',byte2_6,'\n\n\n',
      byte3_6,'\n\n',byte4_6,'\n\n\n',
      byte5_6,'\n\n',byte6_6,'\n\n\n',
      byte7_6,'\n\n',byte8_6,'\n\n\n',
      byte9_6,'\n\n',byte10_6)

[6 6 6] 

 [168 168 168] 


 [5 5 5] 

 [128 110  79] 


 [5 5 6] 

 [181 233  14] 


 [6 6 6] 

 [ 16  86 155] 


 [4 4 4] 

 [26 26 26]


In [686]:
# Send all path points to both servos
x = 1
for i in range(len(path6_full)):
    WriteByte(ser, int(byte1_6[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_6[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_6[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_6[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_6[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_6[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_6[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_6[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_6[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_6[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_6[i]*256 + byte2_6[i]}  "
          f"th2={byte3_6[i]*256 + byte4_6[i]}  "
          f"th3={byte5_6[i]*256 + byte6_6[i]}  "
          f"th4={byte7_6[i]*256 + byte8_6[i]}  "
          f"th5={byte9_6[i]*256 + byte10_6[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1704  th2=1408  th3=1461  th4=1552  th5=1050
Step  1:  th1=1704  th2=1390  th3=1513  th4=1622  th5=1050
Step  2:  th1=1704  th2=1359  th3=1550  th4=1691  th5=1050


## path 7

In [687]:
#define arrays to hold the four bytes
byte1_7 = np.zeros(len(path7_full), dtype=int)
byte2_7 = np.zeros(len(path7_full), dtype=int)
byte3_7 = np.zeros(len(path7_full), dtype=int)
byte4_7 = np.zeros(len(path7_full), dtype=int)
byte5_7 = np.zeros(len(path7_full), dtype=int)
byte6_7 = np.zeros(len(path7_full), dtype=int)
byte7_7 = np.zeros(len(path7_full), dtype=int)
byte8_7 = np.zeros(len(path7_full), dtype=int)
byte9_7 = np.zeros(len(path7_full), dtype=int)
byte10_7 = np.zeros(len(path7_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path7_full)):
    byte1_7[i], byte2_7[i] = break_into_two(path7_full[i,0])
    byte3_7[i], byte4_7[i] = break_into_two(path7_full[i,1])
    byte5_7[i], byte6_7[i] = break_into_two(path7_full[i,2])
    byte7_7[i], byte8_7[i] = break_into_two(path7_full[i,3])
    byte9_7[i], byte10_7[i] = break_into_two(path7_full[i,4])

print(byte1_7,'\n\n',byte2_7,'\n\n\n',
      byte3_7,'\n\n',byte4_7,'\n\n\n',
      byte5_7,'\n\n',byte6_7,'\n\n\n',
      byte7_7,'\n\n',byte8_7,'\n\n\n',
      byte9_7,'\n\n',byte10_7)

[6 6] 

 [168 168] 


 [5 5] 

 [ 79 128] 


 [6 5] 

 [ 14 181] 


 [6 6] 

 [155  16] 


 [5 5] 

 [220 220]


In [688]:
# Send all path points to both servos
x = 1
for i in range(len(path7_full)):
    WriteByte(ser, int(byte1_7[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_7[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_7[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_7[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_7[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_7[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_7[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_7[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_7[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_7[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_7[i]*256 + byte2_7[i]}  "
          f"th2={byte3_7[i]*256 + byte4_7[i]}  "
          f"th3={byte5_7[i]*256 + byte6_7[i]}  "
          f"th4={byte7_7[i]*256 + byte8_7[i]}  "
          f"th5={byte9_7[i]*256 + byte10_7[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1704  th2=1359  th3=1550  th4=1691  th5=1500
Step  1:  th1=1704  th2=1408  th3=1461  th4=1552  th5=1500


## path 8

In [702]:
#define arrays to hold the four bytes
byte1_8 = np.zeros(len(path8_full), dtype=int)
byte2_8 = np.zeros(len(path8_full), dtype=int)
byte3_8 = np.zeros(len(path8_full), dtype=int)
byte4_8 = np.zeros(len(path8_full), dtype=int)
byte5_8 = np.zeros(len(path8_full), dtype=int)
byte6_8 = np.zeros(len(path8_full), dtype=int)
byte7_8 = np.zeros(len(path8_full), dtype=int)
byte8_8 = np.zeros(len(path8_full), dtype=int)
byte9_8 = np.zeros(len(path8_full), dtype=int)
byte10_8 = np.zeros(len(path8_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path8_full)):
    byte1_8[i], byte2_8[i] = break_into_two(path8_full[i,0])
    byte3_8[i], byte4_8[i] = break_into_two(path8_full[i,1])
    byte5_8[i], byte6_8[i] = break_into_two(path8_full[i,2])
    byte7_8[i], byte8_8[i] = break_into_two(path8_full[i,3])
    byte9_8[i], byte10_8[i] = break_into_two(path8_full[i,4])

print(byte1_8,'\n\n',byte2_8,'\n\n\n',
      byte3_8,'\n\n',byte4_8,'\n\n\n',
      byte5_8,'\n\n',byte6_8,'\n\n\n',
      byte7_8,'\n\n',byte8_8,'\n\n\n',
      byte9_8,'\n\n',byte10_8)

[6 6 5] 

 [168  77 220] 


 [5 5 5] 

 [128 190 220] 


 [5 5 5] 

 [181 222 220] 


 [6 5 5] 

 [ 16 251 220] 


 [5 5 5] 

 [220 220 220]


In [703]:
# Send all path points to both servos
x = 1
for i in range(len(path8_full)):
    WriteByte(ser, int(byte1_8[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_8[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_8[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_8[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_8[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_8[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_8[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_8[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_8[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_8[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_8[i]*256 + byte2_8[i]}  "
          f"th2={byte3_8[i]*256 + byte4_8[i]}  "
          f"th3={byte5_8[i]*256 + byte6_8[i]}  "
          f"th4={byte7_8[i]*256 + byte8_8[i]}  "
          f"th5={byte9_8[i]*256 + byte10_8[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1704  th2=1408  th3=1461  th4=1552  th5=1500
Step  1:  th1=1613  th2=1470  th3=1502  th4=1531  th5=1500
Step  2:  th1=1500  th2=1500  th3=1500  th4=1500  th5=1500


## full path

In [718]:
hold_count = 1
hold4close = np.tile(path2_full[0,:],(hold_count,1))
hold4open = np.tile(path7_full[0,:],(hold_count,1))

path_full = np.vstack([path1_full,
                       hold4close,
                       path2_full,
                       path3_full,
                       path4_full,
                       path5_full,
                       path6_full,
                       hold4open,
                       path7_full,
                       path8_full])

print('full path:',path_full)

full path: [[1268.54860095 1401.21991825 1451.07408878 1549.85417053 1500.        ]
 [1268.54860095 1383.68819437 1503.70136116 1620.01316679 1500.        ]
 [1268.54860095 1352.74546722 1540.82589554 1688.08042832 1500.        ]
 [1268.54860095 1352.74546722 1540.82589554 1688.08042832 1050.        ]
 [1268.54860095 1352.74546722 1540.82589554 1688.08042832 1050.        ]
 [1268.54860095 1401.21991825 1451.07408878 1549.85417053 1050.        ]
 [1268.54860095 1401.21991825 1451.07408878 1549.85417053 1050.        ]
 [1187.48046778 1488.22906075 1555.09993508 1566.87087433 1050.        ]
 [1187.48046778 1488.22906075 1555.09993508 1566.87087433 1050.        ]
 [1224.88249738 1524.5487524  1590.58832041 1566.03956802 1050.        ]
 [1274.68500685 1554.90101449 1616.63834536 1561.73733086 1050.        ]
 [1339.21895444 1575.74441406 1632.65764524 1556.91323118 1050.        ]
 [1416.91528713 1583.27137128 1638.07529813 1554.80392686 1050.        ]
 [1500.         1575.74441406 1632.65764

In [719]:
#define arrays to hold the four bytes
byte1 = np.zeros(len(path_full), dtype=int)
byte2 = np.zeros(len(path_full), dtype=int)
byte3 = np.zeros(len(path_full), dtype=int)
byte4 = np.zeros(len(path_full), dtype=int)
byte5 = np.zeros(len(path_full), dtype=int)
byte6 = np.zeros(len(path_full), dtype=int)
byte7 = np.zeros(len(path_full), dtype=int)
byte8 = np.zeros(len(path_full), dtype=int)
byte9 = np.zeros(len(path_full), dtype=int)
byte10 = np.zeros(len(path_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path_full)):
    byte1[i], byte2[i] = break_into_two(path_full[i,0])
    byte3[i], byte4[i] = break_into_two(path_full[i,1])
    byte5[i], byte6[i] = break_into_two(path_full[i,2])
    byte7[i], byte8[i] = break_into_two(path_full[i,3])
    byte9[i], byte10[i] = break_into_two(path_full[i,4])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4,'\n\n\n',byte5,'\n\n',byte6,'\n\n\n',byte7,'\n\n',byte8,'\n\n\n',byte9,'\n\n',byte10)

[4 4 4 4 4 4 4 4 4 4 4 5 5 5 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 5] 

 [244 244 244 244 244 244 244 163 163 200 250  59 136 220  42 108 161 203
 203 194 185 176 168 168 168 168 168 168 168 168  77 220] 


 [5 5 5 5 5 5 5 5 5 5 6 6 6 6 6 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5] 

 [121 103  72  72  72 121 121 208 208 244  18  39  47  39  18 244 208 168
 168 158 149 139 128 128 110  79  79  79 128 128 190 220] 


 [5 5 6 6 6 5 5 6 6 6 6 6 6 6 6 6 6 5 5 5 5 5 5 5 5 6 6 6 5 5 5 5] 

 [171 223   4   4   4 171 171  19  19  54  80  96 102  96  80  54  19 230
 230 219 207 194 181 181 233  14  14  14 181 181 222 220] 


 [6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 5 5] 

 [ 13  84 152 152 152  13  13  30  30  30  25  20  18  20  25  30  30  26
  26  24  22  19  16  16  86 155 155 155  16  16 251 220] 


 [5 5 5 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5] 

 [220 220 220  26  26  26  26  26  26  26  26  26  26  26  26  26  26  26
  26  26  26  26  26  26  26  26 220 220 2

In [720]:
# Send all path points to both servos
x = 1
for i in range(len(path_full)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1[i]*256 + byte2[i]}  "
          f"th2={byte3[i]*256 + byte4[i]}  "
          f"th3={byte5[i]*256 + byte6[i]}  "
          f"th4={byte7[i]*256 + byte8[i]}  "
          f"th5={byte9[i]*256 + byte10[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1268  th2=1401  th3=1451  th4=1549  th5=1500
Step  1:  th1=1268  th2=1383  th3=1503  th4=1620  th5=1500
Step  2:  th1=1268  th2=1352  th3=1540  th4=1688  th5=1500
Step  3:  th1=1268  th2=1352  th3=1540  th4=1688  th5=1050
Step  4:  th1=1268  th2=1352  th3=1540  th4=1688  th5=1050
Step  5:  th1=1268  th2=1401  th3=1451  th4=1549  th5=1050
Step  6:  th1=1268  th2=1401  th3=1451  th4=1549  th5=1050
Step  7:  th1=1187  th2=1488  th3=1555  th4=1566  th5=1050
Step  8:  th1=1187  th2=1488  th3=1555  th4=1566  th5=1050
Step  9:  th1=1224  th2=1524  th3=1590  th4=1566  th5=1050
Step 10:  th1=1274  th2=1554  th3=1616  th4=1561  th5=1050
Step 11:  th1=1339  th2=1575  th3=1632  th4=1556  th5=1050
Step 12:  th1=1416  th2=1583  th3=1638  th4=1554  th5=1050
Step 13:  th1=1500  th2=1575  th3=1632  th4=1556  th5=1050
Step 14:  th1=1578  th2=1554  th3=1616  th4=1561  th5=1050
Step 15:  th1=1644  th2=1524  th3=1590  th4=1566  th5=1050
Step 16:  th1=1697  th2=1488  th3=1555  th4=1566  th5=10